# Otoscope data augmentation (WCT2 pipeline) -- stage 2

Builds the augmented training datasets used in the paper. This is stage 2 of
the AI pipeline: for the `roi` variant, the eardrum ROI cropping (**C** in the
paper) has **already been applied upstream** by `../1_roi_crop_sam2/` -- the
`roi` content split consumed here is its output. The `raw` variant uses the
uncropped frames.

Each training sample then passes through:

1. **S** - WCT2 photorealistic style transfer (decoder-only, `cat5`, 512 px) -- the paper's S
2. **P** - Albumentations `ReplayCompose` (HFlip / Rotate +-180 / GaussianBlur / MotionBlur / RGBShift / RandomBrightness) -- applied in every configuration
3. **Resize** to 256x256 (always), then
4. **B** - ellipse-gradient edge multiply with Gaussian-sampled strength and gamma -- the paper's EB

Operators are sampled per job and applied strictly in the order
**S -> P -> resize -> B** (`pipeline.process_one_image_with_combo`; the edge
mask follows P's geometric transforms so the vignette tracks the rotated
content). Four dataset versions (`p`, `sp`, `pb`, `spb`) come from varying
which operators are allowed.

**Naming note.** `roi` (content split, e.g. `f2_roi/`) and `crop` (output
dataset suffix, e.g. `dataset_spb_f2_crop/`) both denote the paper's C
operator; the names follow the paper's original on-disk artifacts and are kept
verbatim so the outputs match the bundled weights and `train.py` expectations.

The logic lives in `pipeline.py` / `model.py` / `utils/`; this notebook only
sets paths and runs. See `README.md` for the full parameter table.

## 1. Mode and configuration

- **`demo`** runs on the small sample under `../demo_data/` (CPU, a few minutes).
- **`real`** reproduces the paper datasets from `OTOSCOPE_DATA_ROOT`, which must
  hold the content splits (`f2_roi/train`, `f2_raw/train`) and an otoscope
  **style pool** kept disjoint from the otoscope test set. Point `style_pool`
  at your cleaned pool; section 3.5 verifies disjointness by content hash
  before any data is generated. A CUDA GPU is expected.

`CONTENT_VARIANT` selects `roi` (the output of `../1_roi_crop_sam2/`) or `raw`
(uncropped frames). This cell reports whether the inputs were found.

In [ ]:
import os

MODE = "demo"                 # "demo" or "real"
CONTENT_VARIANT = "roi"       # "roi" (output of ../1_roi_crop_sam2/) or "raw" (uncropped frames)

if MODE == "demo":
    DATA_ROOT  = "../demo_data/augmentation"
    SRC_DIR    = os.path.join(DATA_ROOT, f"content_{CONTENT_VARIANT}")
    STYLES_DIR = os.path.join(DATA_ROOT, "styles")
    OUT_ROOT   = "./demo_output"
    TARGET_PER_CLASS = 8      # tiny; the paper uses 20000 per class
elif MODE == "real":
    DATA_ROOT  = os.environ.get("OTOSCOPE_DATA_ROOT", "../real_data")
    SRC_DIR    = os.path.join(DATA_ROOT, f"f2_{CONTENT_VARIANT}", "train")
    # Style pool: otoscope style targets, kept DISJOINT from the otoscope test
    # set. Style transfer copies pool appearance into the training data, so any
    # image shared with the test set is a leak; section 3.5 checks this by
    # content hash. Point this at your cleaned pool.
    _cands = [os.path.join(DATA_ROOT, "style_pool"),
              os.path.join(DATA_ROOT, "__style_pool_clean"),
              os.path.join(DATA_ROOT, "..", "style_pool")]
    STYLES_DIR = next((c for c in _cands if os.path.isdir(c)), _cands[0])
    OUT_ROOT   = DATA_ROOT     # version folders are written next to the inputs
    TARGET_PER_CLASS = 20000
else:
    raise ValueError("MODE must be 'demo' or 'real'")

# Folder suffix used by the paper: ROI -> _crop, RAW -> _raw
OUT_SUFFIX = "crop" if CONTENT_VARIANT == "roi" else "raw"

CACHE_DIR       = f"./cache_{CONTENT_VARIANT}"
EDGE_CACHE_PATH = os.path.join(CACHE_DIR, "edge_params.json")
EDGE_MASKS_DIR  = os.path.join(CACHE_DIR, "masks_npy")
MODEL_PATH      = "./model_checkpoints"

print(f"MODE={MODE}  CONTENT_VARIANT={CONTENT_VARIANT}  TARGET_PER_CLASS={TARGET_PER_CLASS}")
for label, path, kind in [("SRC_DIR   ", SRC_DIR, "content split"),
                          ("STYLES_DIR", STYLES_DIR, "style pool")]:
    mark = "found" if os.path.isdir(path) else "NOT FOUND"
    print(f"{label}: {path}  [{kind}: {mark}]")
print("OUT_ROOT  :", OUT_ROOT)

## 2. Imports

In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pipeline import (
    DEFAULT_SETUP_ARGS,
    StyleTransferEngine,
    collect_style_pool,
    compute_ellipse_params_once,
    imread_rgb,
    precompute_edge_cache,
    process_one_image_with_combo,
    run_augmentation_random_ops,
)


def show_row(images, titles, size=2.6):
    """Small helper for the preview cells: one row of images."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Collect source files per class

The content split is one flat directory of PNGs whose file names contain the
class label.

In [ ]:
src_files = sorted(os.listdir(SRC_DIR))
print(f"Found {len(src_files)} source files in {SRC_DIR}")

class_names = ["NORMAL", "PERFORATION", "RETRACTION", "TYMPANOSCLEROSIS"]
buckets = {name: [] for name in class_names}
for fn in src_files:
    for name in class_names:
        if name in fn:
            buckets[name].append(os.path.join(SRC_DIR, fn))
            break

list_of_lists = [buckets[n] for n in class_names]
for n, paths in zip(class_names, list_of_lists):
    print(f"  {n:>16s}: {len(paths)}")
print(f"Total: {sum(len(v) for v in list_of_lists)}")

### A look at the inputs

One content frame per class (endoscope domain) and the first style targets
(otoscope domain). The augmentation maps the otoscope appearance onto the
endoscope content.

In [ ]:
style_files = collect_style_pool(STYLES_DIR) or []
print(f"Style pool: {len(style_files)} images")

show_row([imread_rgb(paths[0]) for paths in list_of_lists],
         [f"content: {n}" for n in class_names])
show_row([imread_rgb(p) for p in style_files[:4]],
         [f"style {i}" for i in range(min(4, len(style_files)))])

## 3.5 Reproducibility and test-set leak check

Two safeguards run before any data is generated:

- **Seed.** Each training sample draws its style image from the global RNG, so
  the seed is fixed here (the paper uses `1337`) to make the style assignments
  reproducible across runs.
- **Leak check.** The style pool must be disjoint from the otoscope test set —
  otherwise style transfer would paint test-set appearance into the training
  data. Pool and test images are compared by content hash (robust to
  renaming); any overlap raises an error and stops the notebook.

In [ ]:
import hashlib
from pathlib import Path

# --- Seed: fix the global RNG the style selection draws from (paper: 1337) ---
AUG_SEED = 1337
random.seed(AUG_SEED)
print(f"random seed = {AUG_SEED}")

# --- Leak check: style pool must be disjoint from the otoscope test set ---
if MODE == "demo":
    TESTSET_DIR = "../demo_data/classification/otoscope_sample"
else:
    TESTSET_DIR = os.path.join(DATA_ROOT, "otoscope")   # otoscope test set

def _content_hashes(d):
    out = {}
    for p in Path(d).rglob("*"):
        if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".bmp"):
            out[hashlib.md5(p.read_bytes()).hexdigest()] = p.name
    return out

pool_hashes = _content_hashes(STYLES_DIR)
test_hashes = _content_hashes(TESTSET_DIR) if os.path.isdir(TESTSET_DIR) else {}
overlap = set(pool_hashes) & set(test_hashes)

print(f"style pool: {len(pool_hashes)} images   test set: {len(test_hashes)} images")
if not test_hashes:
    print(f"(test set not found at {TESTSET_DIR}; leak check skipped)")
elif overlap:
    shared = [test_hashes[h] for h in list(overlap)[:5]]
    raise AssertionError(
        f"LEAK: {len(overlap)} style-pool images are also in the test set "
        f"(e.g. {shared}). Use a style pool disjoint from the test set.")
else:
    print("leak check OK: style pool and test set are disjoint")

## 4. Pre-compute ellipse-gradient edge masks (for B)

Ellipse parameters are fit once per content image and cached (JSON + `.npy`
masks), so the augmentation runner does not refit them per job.

In [ ]:
_ = precompute_edge_cache(
    list_of_lists=list_of_lists,
    cache_json_path=EDGE_CACHE_PATH,
    save_masks_dir=EDGE_MASKS_DIR,
    thresh1=30,
    morph_kernel_size=15,
    fade_ratio=0.20,
    min_contour_points=5,
    min_area_px=50,
)

### What the edge mask looks like

The B operator multiplies the image by an ellipse-gradient mask fitted to the
visible field of view, reproducing the otoscope's peripheral intensity
falloff. Below: a content frame, its fitted mask, and the plain
image-times-mask product (illustration only -- the real B step samples a
strength in [0.6, 1.0] and a gamma in [0.8, 1.2] per job, shown in section 5).

In [ ]:
sample_path = list_of_lists[0][0]
sample_img  = imread_rgb(sample_path)
_, sample_mask = compute_ellipse_params_once(
    sample_img, thresh1=30, morph_kernel_size=15, fade_ratio=0.20)

masked = (sample_img.astype(np.float32) * sample_mask[..., None]).astype(np.uint8)
show_row([sample_img, sample_mask, masked],
         ["content", "ellipse-gradient mask", "image x mask (strength 1.0)"], size=3.0)

## 5. One sample through the pipeline: S -> P -> resize -> B

The exact per-job function (`pipeline.process_one_image_with_combo`) run three
times on the same content/style pair with growing operator combos. This is the
same code path the full runs in section 6 use -- nothing is re-implemented
here. P is stochastic, so the middle panels vary between executions; every
sampled decision is recorded in the metadata CSV during the real runs.

In [ ]:
engine = StyleTransferEngine(
    mode="wct2", device="cuda", model_path=MODEL_PATH,
    option_unpool="cat5", transfer_at={"decoder"}, image_size=512, alpha=1.0,
)

content_path = list_of_lists[0][0]
style_path   = style_files[0]
STEP_DIR     = Path("./demo_output/_step_through")

stage_imgs, stage_titles = [imread_rgb(content_path), imread_rgb(style_path)], ["content", "style target"]
for i, combo in enumerate(["s", "sp", "spb"]):
    _, meta = process_one_image_with_combo(
        content_path=content_path, class_idx=0, class_name=class_names[0],
        out_root=STEP_DIR, engine=engine, style_pool=[style_path],
        rng=random.Random(1337), final_size=(256, 256),
        jpeg_ext=".jpg", quality_jpg=95, png_level=3,
        variant_idx=i, combo=combo,
        setup_args=DEFAULT_SETUP_ARGS,
        edge_strength_range=(0.6, 1.0), edge_gamma_range=(0.8, 1.2),
    )
    stage_imgs.append(imread_rgb(meta["out_path"]))
    stage_titles.append({"s": "S (style)", "sp": "S + P", "spb": "S + P + B (final)"}[combo])

show_row(stage_imgs, stage_titles, size=2.8)

## 6. Dataset version recipes

Each entry pairs the allowed-ops string with per-op Bernoulli probabilities
and an output tag -- identical to the paper runs (`p` is always on; `s` fires
with 0.9, `b` with 0.85).

The call below passes the same keyword arguments as the paper's original run
scripts. One documentation note: `run_augmentation_random_ops` accepts
`prob_hflip / prob_vflip / prob_rotate` but does not forward them to the
per-job replay builder, so the effective values are the builder defaults --
HFlip 0.5, Rotate 0.9 (+-180 deg) -- exactly as recorded in the paper's
metadata CSVs. The kwargs are kept verbatim for fidelity with the original
scripts.

In [ ]:
ver_p   = ("p",   {"p": 1.0},                             "dataset_p_f2")
ver_sp  = ("sp",  {"p": 1.0, "s": 0.9},                   "dataset_sp_f2")
ver_pb  = ("pb",  {"p": 1.0, "b": 0.85},                  "dataset_pb_f2")
ver_spb = ("spb", {"p": 1.0, "s": 0.9, "b": 0.85},        "dataset_spb_f2")


def make_paths(tag):
    out_dir  = f"{OUT_ROOT}/{tag}_{OUT_SUFFIX}"
    csv_path = f"{OUT_ROOT}/{tag}_{OUT_SUFFIX}_metadata.csv"
    return out_dir, csv_path


def run_version(version):
    allowed_ops, op_probs, tag = version
    out_dir, csv_path = make_paths(tag)
    return run_augmentation_random_ops(
        list_of_lists=list_of_lists,
        class_names=class_names,
        out_dir=out_dir,
        allowed_ops=allowed_ops,
        op_probs=op_probs,
        target_per_class=TARGET_PER_CLASS,
        styles_dir=STYLES_DIR,

        engine_mode="wct2",
        wct2_model_path=MODEL_PATH,
        wct2_option_unpool="cat5",
        wct2_transfer_at={"decoder"},
        wct2_image_size=512,
        wct2_alpha=1.0,

        final_h=256, final_w=256,
        setup_args=DEFAULT_SETUP_ARGS,
        prob_hflip=0.5, prob_vflip=0.0,
        prob_rotate=0.99, rotate_limit_deg=180,

        edge_cache_json=EDGE_CACHE_PATH,
        edge_masks_dir=EDGE_MASKS_DIR,
        use_edge_ram=True,
        edge_ram_preload=False,
        edge_strength_range=(0.6, 1.0),
        edge_gamma_range=(0.8, 1.2),

        csv_path=csv_path,
    )

### Run the versions

Each `run_version` writes `<OUT_ROOT>/<tag>_<suffix>/<class>/*.jpg` plus a
metadata CSV. In `real` mode each call produces ~80 000 images (20 000 per
class) -- comment out the versions you do not need.

In [ ]:
df_p   = run_version(ver_p)
df_sp  = run_version(ver_sp)
df_pb  = run_version(ver_pb)
df_spb = run_version(ver_spb)

## 7. Inspect the outputs

A few images from the full `spb` version and the head of its metadata CSV.
The CSV records every per-image decision (operator combo, style path, the
JSON-encoded Albumentations replay, ellipse-fit parameters, sampled vignette
strength/gamma, WCT2 settings), which makes each output image reproducible.

In [ ]:
import pandas as pd

out_dir, csv_path = make_paths("dataset_spb_f2")
out_paths = sorted(Path(out_dir).rglob("*.jpg"))[:6]
show_row([imread_rgb(str(p)) for p in out_paths],
         [p.parent.name for p in out_paths], size=2.2)

meta_df = pd.read_csv(csv_path)
print(f"{csv_path}: {len(meta_df)} rows")
meta_df[["out_path", "combo_tag", "apply_style", "apply_proc", "apply_border",
         "edge_strength", "edge_gamma"]].head(8)